# BirdCLEF+ 2026 — Inference Notebook (提出用)

このノートブックは BirdCLEF 2026 の **提出用推論ノートブック** です。

## 使い方
1. Datasets に以下を追加:
   - `birdclef-2026` (competition data)
   - `birdclef-2026-models` (学習済みモデルの `.pth` + `label_encoder.pkl`)
2. Accelerator: **None (CPU)** に設定
3. Internet access: **OFF** に設定
4. 「Submit」で提出

## 設計の要点
- CPU 90分制限に対応するため**バッチ推論**を採用
- fold モデルを上位 N 個に絞って時間短縮
- `torch.no_grad()` + `float32` で CPU 推論を安定化

In [ ]:
# ============================================================
# 設定 — ここだけ変更すれば動く
# ============================================================
COMP_DATA     = '/kaggle/input/birdclef-2026'
MODELS_DIR    = '/kaggle/input/birdclef-2026-models'   # 学習済みモデルのパス
OUTPUT_DIR    = '/kaggle/working'

# モデルアーキテクチャ (学習時と一致させること)
MODEL_NAME    = 'tf_efficientnet_b3_ns'
USE_GEM       = True
GEM_P         = 3.0

# 音声パラメータ (学習時と一致させること)
SAMPLE_RATE   = 32000
DURATION      = 5          # 秒
N_MELS        = 224
N_FFT         = 2048
HOP_LENGTH    = 256
FMIN          = 50
FMAX          = 16000

# 推論パラメータ
INFERENCE_BATCH_SIZE = 32  # 複数ウィンドウを一括処理 (速度向上)
OVERLAP_SECONDS      = 2.5 # スライドウィンドウのオーバーラップ (秒)
MAX_FOLD_MODELS      = 3   # 使用するfoldモデルの上限 (時間短縮)

# ImageNet 正規化定数
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

In [ ]:
import collections
import os
import pickle
import time
import warnings

import librosa
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('PyTorch:', torch.__version__)
print('Device: CPU (no GPU for submission)')
print('Timm:', timm.__version__)

In [ ]:
# ============================================================
# モデル定義 (src/model.py と同じ)
# ============================================================

class GeMPooling(nn.Module):
    """Generalized Mean Pooling — p=1 が GAP、p→∞ が GMP に相当。"""
    def __init__(self, p: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return (
            F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), 1)
            .pow(1.0 / self.p)
            .flatten(1)
        )


class BirdCLEFModel(nn.Module):
    def __init__(self, model_name, num_classes, use_gem=True, gem_p=3.0, pretrained=False):
        super().__init__()
        self.use_gem = use_gem
        if use_gem:
            self.backbone = timm.create_model(
                model_name, pretrained=pretrained,
                num_classes=0, global_pool='', in_chans=3)
            self.pool = GeMPooling(p=gem_p)
        else:
            self.backbone = timm.create_model(
                model_name, pretrained=pretrained,
                num_classes=0, in_chans=3)
            self.pool = None
        self.head = nn.Linear(self.backbone.num_features, num_classes)

    def forward(self, x):
        if self.use_gem:
            return self.head(self.pool(self.backbone.forward_features(x)))
        return self.head(self.backbone(x))

    @torch.no_grad()
    def predict(self, x):
        return torch.sigmoid(self.forward(x))


print('モデル定義 OK')

In [ ]:
# ============================================================
# 音声処理ユーティリティ (src/utils.py と同じロジック)
# ============================================================

def audio_to_melspec(waveform: np.ndarray) -> np.ndarray:
    """波形 → 正規化済みメルスペクトログラム (n_mels, time_steps)."""
    mel = librosa.feature.melspectrogram(
        y=waveform, sr=SAMPLE_RATE,
        n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH,
        fmin=FMIN, fmax=FMAX,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_min, mel_max = mel_db.min(), mel_db.max()
    if mel_max - mel_min > 1e-6:
        return ((mel_db - mel_min) / (mel_max - mel_min)).astype(np.float32)
    return np.zeros_like(mel_db, dtype=np.float32)


def window_to_tensor(window: np.ndarray) -> torch.Tensor:
    """5秒波形ウィンドウ → (3, n_mels, time_steps) 正規化済みテンソル."""
    mel = audio_to_melspec(window)                          # (n_mels, T)
    t   = torch.from_numpy(mel).unsqueeze(0).repeat(3, 1, 1)  # (3, n_mels, T)
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t - mean) / std                                 # (3, n_mels, T)


print('音声処理ユーティリティ OK')

In [ ]:
# ============================================================
# ラベルエンコーダ & モデルの読み込み
# ============================================================

device = torch.device('cpu')

# Label encoder
le_path = os.path.join(MODELS_DIR, 'label_encoder.pkl')
with open(le_path, 'rb') as f:
    le = pickle.load(f)
num_classes = len(le.classes_)
print(f'クラス数: {num_classes}')

# Fold モデルを読み込む（val_score 降順でソートし上位 MAX_FOLD_MODELS 個を使用）
ckpt_files = sorted([
    os.path.join(MODELS_DIR, f)
    for f in os.listdir(MODELS_DIR)
    if f.startswith('fold') and f.endswith('.pth')
])
print(f'チェックポイント候補: {len(ckpt_files)} 個')

# val_score でソートして上位を選択
ckpt_meta = []
for p in ckpt_files:
    ckpt = torch.load(p, map_location='cpu')
    ckpt_meta.append((p, ckpt.get('val_score', 0.0)))
ckpt_meta.sort(key=lambda x: x[1], reverse=True)
selected = ckpt_meta[:MAX_FOLD_MODELS]

models = []
for path, score in selected:
    m = BirdCLEFModel(MODEL_NAME, num_classes, use_gem=USE_GEM, gem_p=GEM_P)
    ckpt = torch.load(path, map_location='cpu')
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    models.append(m)
    print(f'  Loaded {os.path.basename(path)} — val_cmap={score:.4f}')

print(f'\n使用モデル数: {len(models)}')

In [ ]:
# ============================================================
# サンプル提出ファイル & 種列の読み込み
# ============================================================

sample_sub = pd.read_csv(f'{COMP_DATA}/sample_submission.csv')
species_cols = [c for c in sample_sub.columns if c != 'row_id']
print(f'提出行数: {len(sample_sub)}')
print(f'種列数:   {len(species_cols)}')

# 学習時の label encoder に存在する種だけマッピング
species_to_idx = {}
for sp in species_cols:
    try:
        species_to_idx[sp] = int(le.transform([sp])[0])
    except ValueError:
        pass
print(f'LEncoder にある種: {len(species_to_idx)} / {len(species_cols)}')

In [ ]:
# ============================================================
# バッチ推論 (CPU 最適化)
# ============================================================

@torch.no_grad()
def predict_batch(tensors: list) -> np.ndarray:
    """テンソルのリストをバッチ推論し (N, num_classes) の確率配列を返す。"""
    batch = torch.stack(tensors, dim=0)   # (N, 3, H, W)
    all_preds = []
    for m in models:
        probs = m.predict(batch)           # (N, num_classes)
        all_preds.append(probs.numpy())
    return np.mean(all_preds, axis=0)      # (N, num_classes)


# テストサウンドスケープを列挙
test_dir = f'{COMP_DATA}/test_soundscapes'
test_files = sorted(f for f in os.listdir(test_dir) if f.lower().endswith('.ogg'))
print(f'テストファイル数: {len(test_files)}')

window_samples  = SAMPLE_RATE * DURATION
overlap_samples = int(SAMPLE_RATE * OVERLAP_SECONDS)
step_samples    = window_samples - overlap_samples

# bucket_key → 予測リスト
bucket_preds = collections.defaultdict(list)

t_start = time.time()

for fname in tqdm(test_files, desc='Inference'):
    file_path = os.path.join(test_dir, fname)
    stem = os.path.splitext(fname)[0]

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        waveform, _ = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)

    total_samples = len(waveform)
    if total_samples == 0:
        continue

    # ウィンドウとそのバケットキーを収集
    windows_batch = []   # テンソルのバッファ
    bucket_keys   = []   # 対応するバケットキー

    start = 0
    while start < total_samples:
        window = waveform[start : start + window_samples]
        if len(window) < window_samples:
            window = np.pad(window, (0, window_samples - len(window)))

        bucket_idx   = start // window_samples
        bucket_start = bucket_idx * DURATION
        row_id = f'{stem}_{bucket_start}'

        windows_batch.append(window_to_tensor(window))
        bucket_keys.append(row_id)

        # バッファが満杯になったらバッチ推論
        if len(windows_batch) >= INFERENCE_BATCH_SIZE:
            batch_probs = predict_batch(windows_batch)  # (N, num_classes)
            for bkey, probs in zip(bucket_keys, batch_probs):
                bucket_preds[bkey].append(probs)
            windows_batch, bucket_keys = [], []

        start += step_samples

    # 残りのウィンドウを処理
    if windows_batch:
        batch_probs = predict_batch(windows_batch)
        for bkey, probs in zip(bucket_keys, batch_probs):
            bucket_preds[bkey].append(probs)

elapsed = time.time() - t_start
print(f'\n推論完了: {elapsed/60:.1f} 分')
print(f'バケット数: {len(bucket_preds)}')

In [ ]:
# ============================================================
# 提出ファイルの作成
# ============================================================

rows = []
for row_id, pred_list in bucket_preds.items():
    avg = np.mean(pred_list, axis=0)     # 重複ウィンドウの平均
    row = {'row_id': row_id}
    for sp in species_cols:
        row[sp] = float(avg[species_to_idx[sp]]) if sp in species_to_idx else 0.0
    rows.append(row)

submission = pd.DataFrame(rows, columns=['row_id'] + species_cols)

# サンプル提出の順番に合わせる & 欠損行を 0 で埋める
submission = (
    submission
    .set_index('row_id')
    .reindex(sample_sub['row_id'].tolist(), fill_value=0.0)
    .reset_index()
)
if 'index' in submission.columns:
    submission = submission.rename(columns={'index': 'row_id'})

out_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission.to_csv(out_path, index=False)

print(f'提出ファイル: {out_path}')
print(f'  行数: {len(submission)}')
print(f'  列数: {len(submission.columns)}')
submission.head(3)

In [ ]:
# 最終確認
assert len(submission) == len(sample_sub), 'row 数が sample_submission と一致しません!'
assert list(submission.columns) == list(sample_sub.columns), '列が一致しません!'
assert submission.isnull().sum().sum() == 0, 'NaN が含まれています!'
print('提出ファイルの検証 OK!')
print(f'\n予測スコアの統計:')
score_df = submission.drop('row_id', axis=1)
print(f'  平均: {score_df.values.mean():.4f}')
print(f'  最大: {score_df.values.max():.4f}')
print(f'  ゼロ率: {(score_df.values == 0.0).mean():.2%}')